In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/bengaluru_model_data.csv")

print("Dataset shape:", df.shape)
print(df.head())

Dataset shape: (11965, 9)
              area_type                  location  total_sqft  bath  balcony  \
0  Super built-up  Area  Electronic City Phase II      1056.0   2.0      1.0   
1            Plot  Area          Chikka Tirupathi      2600.0   5.0      3.0   
2        Built-up  Area               Uttarahalli      1440.0   2.0      3.0   
3  Super built-up  Area        Lingadheeranahalli      1521.0   3.0      1.0   
4  Super built-up  Area                  Kothanur      1200.0   2.0      1.0   

    price  bhk  price_per_sqft  sqft_per_bhk  
0   39.07    2     3699.810606         528.0  
1  120.00    4     4615.384615         650.0  
2   62.00    3     4305.555556         480.0  
3   95.00    3     6245.890861         507.0  
4   51.00    2     4250.000000         600.0  


In [2]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=["price"])
y = df["price"]

# Remove diagnostic columns, not model inputs
X = X.drop(columns=["price_per_sqft", "sqft_per_bhk"], errors="ignore")

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42
)

print("Training features:", X_train.shape)
print("Testing features:", X_test.shape)

print("Training target:", y_train.shape)
print("Testing target:", y_test.shape)

Training features: (9572, 6)
Testing features: (2393, 6)
Training target: (9572,)
Testing target: (2393,)


In [3]:
# Count locations in training data
location_counts = X_train["location"].value_counts()

# Keep locations having more than 10 records
common_locations = location_counts[location_counts > 10].index

# Replace uncommon training locations with Other
X_train["location"] = X_train["location"].apply(
    lambda x: x if x in common_locations else "Other"
)

# Replace unseen or uncommon testing locations with Other
X_test["location"] = X_test["location"].apply(
    lambda x: x if x in common_locations else "Other"
)

print("Training locations:", X_train["location"].nunique())
print("Testing locations:", X_test["location"].nunique())

Training locations: 188
Testing locations: 182


In [4]:
import joblib

common_locations = list(common_locations)

joblib.dump(common_locations, "../artifacts/common_locations.pkl")

print("Common locations saved:", len(common_locations))

Common locations saved: 187


In [5]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Categorical columns
categorical_columns = ["area_type", "location"]

# Numerical columns
numerical_columns = [
    "total_sqft",
    "bath",
    "balcony",
    "bhk"
]

# Preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_columns),
        ("num", StandardScaler(), numerical_columns)
    ]
)

# Fit on training data, transform both datasets
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("Training data:", X_train_processed.shape)
print("Testing data:", X_test_processed.shape)

Training data: (9572, 196)
Testing data: (2393, 196)


In [6]:
from sklearn.linear_model import LinearRegression

# Create model
model = LinearRegression()

# Train model
model.fit(X_train_processed, y_train)

print("Linear Regression training completed!")

Linear Regression training completed!


In [7]:
# Predict house prices
y_pred = model.predict(X_test_processed)

print("Actual prices:")
print(y_test.head(10).values)

print("\nPredicted prices:")
print(y_pred[:10])

Actual prices:
[ 55.    98.    79.    39.2   65.    93.36  44.    50.   108.    68.2 ]

Predicted prices:
[ 65.09465831 124.05354428  54.76599169  60.07065761  55.88945262
  60.53093787  37.05190121  72.61846374 113.93917874  43.33206689]


In [8]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("MAE:", mae)
print("RMSE:", rmse)
print("R2 Score:", r2)

MAE: 34.854318097202245
RMSE: 77.68998167563664
R2 Score: 0.6182198689962418


In [9]:
from sklearn.tree import DecisionTreeRegressor

model_dt = DecisionTreeRegressor(random_state=42)

model_dt.fit(X_train_processed, y_train)

print("Decision Tree training completed!")

Decision Tree training completed!


In [10]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

y_pred_dt = model_dt.predict(X_test_processed)

print("MAE:", mean_absolute_error(y_test, y_pred_dt))

print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred_dt)))

print("R2 Score:", r2_score(y_test, y_pred_dt))

MAE: 36.16848961957623
RMSE: 98.93724292692343
R2 Score: 0.38083999918790823


In [11]:
print("Linear Regression R2:", r2_score(y_test, y_pred))
print("Decision Tree R2:", r2_score(y_test, y_pred_dt))

Linear Regression R2: 0.6182198689962418
Decision Tree R2: 0.38083999918790823


In [12]:
from sklearn.ensemble import RandomForestRegressor

model_rf = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

model_rf.fit(X_train_processed, y_train)

print("Random Forest training completed!")

Random Forest training completed!


In [13]:
y_pred_rf = model_rf.predict(X_test_processed)

print("MAE:", mean_absolute_error(y_test, y_pred_rf))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred_rf)))
print("R2 Score:", r2_score(y_test, y_pred_rf))

MAE: 30.089350194268892
RMSE: 80.50709730545431
R2 Score: 0.5900304309705214


In [14]:
print("Linear Regression:", r2_score(y_test, y_pred))
print("Decision Tree:", r2_score(y_test, y_pred_dt))
print("Random Forest:", r2_score(y_test, y_pred_rf))

Linear Regression: 0.6182198689962418
Decision Tree: 0.38083999918790823
Random Forest: 0.5900304309705214


In [15]:
from sklearn.ensemble import GradientBoostingRegressor

model_gb = GradientBoostingRegressor(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    random_state=42
)

model_gb.fit(X_train_processed, y_train)

print("Gradient Boosting training completed!")

Gradient Boosting training completed!


In [16]:
y_pred_gb = model_gb.predict(X_test_processed)

print("MAE:", mean_absolute_error(y_test, y_pred_gb))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred_gb)))
print("R2 Score:", r2_score(y_test, y_pred_gb))

MAE: 31.985069620915095
RMSE: 76.96306952509325
R2 Score: 0.625330754823671


In [17]:
print("Linear Regression:", r2_score(y_test, y_pred))
print("Decision Tree:", r2_score(y_test, y_pred_dt))
print("Random Forest:", r2_score(y_test, y_pred_rf))
print("Gradient Boosting:", r2_score(y_test, y_pred_gb))

Linear Regression: 0.6182198689962418
Decision Tree: 0.38083999918790823
Random Forest: 0.5900304309705214
Gradient Boosting: 0.625330754823671


In [18]:
from sklearn.model_selection import GridSearchCV

params = {
    "n_estimators": [100, 200],
    "learning_rate": [0.05, 0.1],
    "max_depth": [2, 3, 4]
}

grid = GridSearchCV(
    GradientBoostingRegressor(random_state=42),
    params,
    cv=3,
    scoring="r2",
    n_jobs=-1
)

grid.fit(X_train_processed, y_train)

print("Best parameters:", grid.best_params_)
print("Best CV R2:", grid.best_score_)

Best parameters: {'learning_rate': 0.05, 'max_depth': 4, 'n_estimators': 200}
Best CV R2: 0.6761631613811382


In [19]:
best_model = grid.best_estimator_

y_pred_best = best_model.predict(X_test_processed)

print("MAE:", mean_absolute_error(y_test, y_pred_best))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred_best)))
print("R2:", r2_score(y_test, y_pred_best))

MAE: 30.79010723283927
RMSE: 75.50056122223421
R2: 0.6394349361501019


In [20]:
import joblib
from pathlib import Path

Path("../artifacts").mkdir(exist_ok=True)

joblib.dump(best_model, "../artifacts/house_price_model.pkl")
joblib.dump(preprocessor, "../artifacts/preprocessor.pkl")

print("Model and preprocessor saved!")

Model and preprocessor saved!
